# NLP Capstone: Production-Ready Abstractive Summarizer
**3MTT Deeptech Ready – NLP Path**  
**Model:** T5 (Text-to-Text Transfer Transformer)  
**Source Article:** *Can 3MTT turn Nigeria into a global tech talent powerhouse?* — TechCabal

---

## NLP Project Workflow
```
1. Problem Definition
2. Data Acquisition       ← we start here (manual)
3. Preprocessing          ← Task 1
4. Model Loading
5. Inference/Generation   ← Task 2
6. Evaluation             ← Task 3
```

---
## Step 0: Environment Check

Before any NLP project, verify your tools are installed. This is your sanity check.

In [ ]:
# Environment check — run this first every time
import importlib

packages = ["transformers", "torch", "rouge_score", "sentencepiece"]
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, '__version__', 'installed')
        print(f"  OK  {pkg} ({version})")
    except ImportError:
        print(f"  MISSING  {pkg} — run: pip install {pkg}")

---
## Task 1: The Text-to-Text Data Pipeline

### Why does T5 need a prefix?

T5 was trained on many tasks at once (summarization, translation, Q&A, etc.).
The only way it knows *which task* you want is the **prefix string**.

| Task | Prefix |
|------|--------|
| Summarization | `"summarize: "` |
| Translation | `"translate English to French: "` |
| Sentiment | `"sst2 sentence: "` |

Skip the prefix → the model guesses the task and gives garbage output.

### Step 1a: Load the raw article (Data Acquisition — Manual)

In [ ]:
# RAW DATA — manually copied from TechCabal
# Source: https://techcabal.com — "Can 3MTT turn Nigeria into a global tech talent powerhouse?"
# Author: Ganiu Oloruntade | Date: October 4, 2024

news_article = """
Will Nigeria's 3 Million Technical Talents Initiative bridge the growing tech talent gap while 
positioning its workforce for global opportunities?

In November 2023, Maryam Shuaibu Aliyu was scrolling through Facebook on her phone at home in Kano 
when she stumbled on a post announcing Nigeria's 3 Million Technical Talents (3MTT) programme.

Despite not knowing how to use a laptop proficiently, Aliyu, a Bayero University graduate, volunteer 
teacher, and mother of two, felt a pull. "I would never have imagined learning a tech skill if I hadn't 
come across the programme," said Aliyu, who was selected as one of 31,270 fellows for the first cohort 
of the 3MTT.

The four-year initiative, launched by the Federal Ministry of Communications, Innovation, and Digital 
Economy, wants to transform Nigeria into a hub for global tech talent. The minister, Bosun Tijani, calls 
it "the largest technology talent accelerator in the world." President Bola Tinubu also referred to it 
during his October 2024 Independence Day speech, outlining it as one of his administration's big plans.

Over 1.7 million people applied to join one of the 12 technical skills offered through the programme: 
software development, UI/UX design, data analysis and visualisation, quality assurance, product 
management, data science, animation, AI/machine learning, cybersecurity, game development, cloud 
computing, and DevOps. 3MTT received applications from almost all of Nigeria's 774 local governments.

The 3MTT programme uses a tiered 1-10-100 structure, starting with a small cohort and scaling 
efficiently while learning from each phase. The first phase (1%) surpassed the 30,000 goal, onboarding 
31,270 fellows, with each cohort lasting three months. The next phase (10%), which began in August 2024, 
will ramp up the number of fellows to 300,000, while the last phase, which will end in 2027, will hit 
the 3 million target (100%).

3MTT reports that it has generated over 3,500 jobs for its fellows, resulting in an 11.21% placement 
rate for Cohort 1. It also claims to have created 4,000 micro-job opportunities. These roles are an 
entry-level mix of full-time, internship, or contract roles at startups, corporates, the public sector.

Bridging the gap between training and employability is the next critical step. The success of 3MTT 
will depend not just on training but on the local ecosystem's ability to use this influx of entry-level 
talent. Tijani believes that more startups should be offering placement opportunities to fellows. 
\"3MTT is a foundational program to get you in. We need the support of the ecosystem.\"

Though 3MTT is opening new doors for thousands of people like Aliyu across Nigeria, the short-term 
gaps — from mismatched expectations to placement bottlenecks — are undeniable. However, the long-term 
potential remains powerful. In the first cohort, 85% of learners rated their likelihood of recommending 
the programme to a friend at 7/10 and above.
"""

# Quick data inspection — always do this before any processing
word_count = len(news_article.split())
char_count = len(news_article)
paragraph_count = len([p for p in news_article.strip().split('\n\n') if p.strip()])

print("=== DATA INSPECTION ===")
print(f"Characters : {char_count:,}")
print(f"Words      : {word_count:,}")
print(f"Paragraphs : {paragraph_count}")
print(f"\nFirst 200 chars (preview):")
print(news_article[:200].strip())

### Step 1b: Preprocessing Function

**What does preprocessing actually do here?**

1. **Strip whitespace** — removes leading/trailing newlines (common in copy-paste text)
2. **Normalize spaces** — collapses double/triple spaces to single space
3. **Prefix** — adds `"summarize: "` so T5 knows the task

> *In a production scraper, you'd also: remove HTML tags, strip author bylines, remove image captions. We'll add those in Phase 2 (scraper).*

In [ ]:
import re

def preprocess_for_t5(text: str) -> str:
    """
    Prepare raw text for T5 summarization.
    
    Steps:
      1. Strip leading/trailing whitespace
      2. Collapse multiple whitespace/newlines into a single space
      3. Prepend the T5 task prefix
    
    Args:
        text: Raw article string
    Returns:
        T5-formatted string ready for tokenization
    """
    # Step 1: Remove leading and trailing whitespace
    cleaned = text.strip()
    
    # Step 2: Replace multiple whitespace characters (including newlines) with single space
    # \s+ matches any whitespace sequence: spaces, tabs, newlines
    cleaned = re.sub(r'\s+', ' ', cleaned)
    
    # Step 3: Prepend the T5 task prefix — THIS IS THE CRITICAL STEP
    t5_input = "summarize: " + cleaned
    
    return t5_input


# ── DELIVERABLE: Before vs After ──────────────────────────────────────────────
print("BEFORE (raw text, first 300 chars):")
print("-" * 50)
print(repr(news_article[:300]))

print("\nAFTER (preprocessed, first 300 chars):")
print("-" * 50)
processed_article = preprocess_for_t5(news_article)
print(repr(processed_article[:300]))

print(f"\nPrefix present: {processed_article.startswith('summarize: ')}")
print(f"Length change : {len(news_article)} chars → {len(processed_article)} chars")

### Step 1c: Tokenization

**Why tokenize?** Neural networks cannot read strings — they only understand numbers.
Tokenization converts text → token IDs.

**Key parameters we set:**
- `max_length=512` — T5-small was trained with a 512-token context window. Longer inputs get cut.
- `truncation=True` — silently cut at 512 tokens instead of raising an error
- `padding='max_length'` — pad shorter inputs to 512 so they batch consistently
- `return_tensors='pt'` — return PyTorch tensors (not plain lists), ready for the model

In [ ]:
from transformers import T5Tokenizer

# Load the tokenizer — this downloads ~4MB of vocabulary files on first run
# 't5-small' is the smallest T5 variant: 60M parameters, fast, good for learning
print("Loading tokenizer...")
tokenizer = T5Tokenizer.from_pretrained('t5-small')
print("Tokenizer loaded.")

# Tokenize the preprocessed text
inputs = tokenizer(
    processed_article,
    max_length=512,
    truncation=True,
    padding='max_length',
    return_tensors='pt'   # 'pt' = PyTorch tensor
)

print("\n=== TOKENIZATION RESULT ===")
print(f"Input shape       : {inputs['input_ids'].shape}")
print(f"  → (batch_size=1, sequence_length=512)")
print(f"\nFirst 10 token IDs: {inputs['input_ids'][0][:10].tolist()}")
print(f"Last  10 token IDs: {inputs['input_ids'][0][-10:].tolist()}")

# Decode first 10 tokens so you can see what they represent
first_tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0][:15].tolist())
print(f"\nFirst 15 tokens decoded: {first_tokens}")
print("  ↑ Notice the prefix tokens and the article beginning")

# Count real tokens vs padding
attention_mask = inputs['attention_mask'][0]
real_tokens = attention_mask.sum().item()
print(f"\nReal tokens (not padding): {int(real_tokens)}/512")
print(f"Padding tokens            : {512 - int(real_tokens)}/512")

---
## Task 2: Decoding Strategies (Inference)

### The key insight: generation is not a single step

The model doesn't produce the whole summary at once. It generates **one token at a time**,
and the **decoding strategy** decides which token to pick next.

| Strategy | How it picks next token | Speed | Quality |
|----------|------------------------|-------|----------|
| **Greedy** | Always picks the single highest-probability token | Fast | Can loop/repeat |
| **Beam Search** | Keeps top-N candidate sequences in parallel, picks the best overall | Slower | More coherent |

Think of it like GPS navigation:
- Greedy = always take the next shortest road (local optimum)
- Beam Search = consider 5 routes simultaneously, pick the best full path (global optimum)

### Load the model

In [ ]:
from transformers import T5ForConditionalGeneration
import torch

# Load T5-small — ~242MB download on first run, cached afterwards
print("Loading T5-small model...")
model = T5ForConditionalGeneration.from_pretrained('t5-small')
model.eval()  # Set to evaluation mode (disables dropout layers used during training)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
inputs_on_device = {k: v.to(device) for k, v in inputs.items()}

print(f"Model loaded on: {device}")
print(f"Model parameters: ~60 million (t5-small)")

### Strategy A: Greedy Search (`num_beams=1`)

In [ ]:
import time

print("Running Strategy A: Greedy Search...")
start = time.time()

with torch.no_grad():  # Disable gradient calculation — not needed for inference, saves memory
    greedy_output = model.generate(
        input_ids=inputs_on_device['input_ids'],
        attention_mask=inputs_on_device['attention_mask'],
        max_new_tokens=80,   # Maximum tokens in the generated summary
        num_beams=1,         # Greedy: only one candidate sequence at a time
    )

greedy_time = time.time() - start

# Decode output token IDs back to text
# skip_special_tokens=True removes <pad>, </s>, etc.
summary_greedy = tokenizer.decode(greedy_output[0], skip_special_tokens=True)

print(f"\n=== STRATEGY A: Greedy Search ===")
print(f"Time: {greedy_time:.2f}s")
print(f"Summary:")
print(summary_greedy)

### Strategy B: Beam Search (`num_beams=5`)

In [ ]:
print("Running Strategy B: Beam Search...")
start = time.time()

with torch.no_grad():
    beam_output = model.generate(
        input_ids=inputs_on_device['input_ids'],
        attention_mask=inputs_on_device['attention_mask'],
        max_new_tokens=80,
        num_beams=5,               # Keep top 5 candidate sequences in parallel
        no_repeat_ngram_size=2,    # Prevent repeating any 2-word phrase (reduces loops)
        early_stopping=True,       # Stop when all beams reach the end token
    )

beam_time = time.time() - start

summary_beam = tokenizer.decode(beam_output[0], skip_special_tokens=True)

print(f"\n=== STRATEGY B: Beam Search ===")
print(f"Time: {beam_time:.2f}s")
print(f"Summary:")
print(summary_beam)

print(f"\n--- Speed comparison ---")
print(f"Greedy : {greedy_time:.2f}s")
print(f"Beam   : {beam_time:.2f}s  ({beam_time/greedy_time:.1f}x slower)")

### Brief Analysis (Task 2 Deliverable)

In [ ]:
analysis = """
ANALYSIS — Greedy vs Beam Search
=================================
Strategy B (Beam Search) produces a more coherent and complete summary.
Beam search evaluates 5 candidate sequences simultaneously, allowing the model
to select a globally better output rather than greedily committing to each
locally optimal token. The no_repeat_ngram_size=2 parameter further prevents
phrase repetition, which is a known failure mode of Strategy A (Greedy Search).
"""
print(analysis)

---
## Task 3: Quantitative Evaluation — ROUGE Metrics

### Why do we need a number?

"This summary sounds good" is subjective and doesn't scale.
In production, models are evaluated against a **Gold Standard** (human-written reference)
using automated metrics.

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) measures **word overlap**:

| Metric | Measures |
|--------|----------|
| **ROUGE-1** | Overlap of individual words (unigrams) |
| **ROUGE-2** | Overlap of word pairs (bigrams) |
| **ROUGE-L** | Longest common subsequence — captures sentence structure |

Scores range 0–1. Higher = more overlap with the human reference.

### Step 3a: Write your Gold Standard

> **Your job:** Read the article and write **your own 1-sentence summary** below.
> This is what a human expert would consider the perfect summary.

In [ ]:
# ── WRITE YOUR GOLD STANDARD HERE ────────────────────────────────────────────
# Edit this string with your own 1-sentence summary of the article

gold_standard = (
    "Nigeria's 3MTT initiative aims to train 3 million technical talents by 2027 "
    "to bridge the digital skills gap, though early results show challenges in "
    "job placement and skills depth despite high enrolment."
)

print("Gold Standard (your human reference):")
print(gold_standard)
print(f"\nWord count: {len(gold_standard.split())}")

### Step 3b: Calculate ROUGE Scores

In [ ]:
from rouge_score import rouge_scorer

# Create a scorer that computes ROUGE-1 and ROUGE-L
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
# use_stemmer=True: treats "training" and "train" as the same word — fairer comparison

# Score beam search summary (Strategy B) against gold standard
scores = scorer.score(
    target=gold_standard,    # Reference (human-written)
    prediction=summary_beam  # Model output (Strategy B)
)

print("=== ROUGE EVALUATION RESULTS ===")
print(f"\nCandidate (Model - Beam Search):")
print(f"  {summary_beam}")
print(f"\nReference (Gold Standard):")
print(f"  {gold_standard}")

print(f"\n{'Metric':<12} {'Precision':>10} {'Recall':>10} {'F1 Score':>10}")
print("-" * 45)
for metric_name, score in scores.items():
    print(f"{metric_name:<12} {score.precision:>10.4f} {score.recall:>10.4f} {score.fmeasure:>10.4f}")

print()
print("Interpretation:")
print("  Precision = of words in model output, how many are in the reference?")
print("  Recall    = of words in the reference, how many did the model capture?")
print("  F1 Score  = harmonic mean of Precision and Recall (the main number to report)")

---
## Summary: What We Built and Learned

| Step | What we did | NLP Concept |
|------|-------------|-------------|
| Data Acquisition | Manually copied TechCabal article | Domain-specific data selection |
| Preprocessing | `preprocess_for_t5()` — clean + prefix | Task-conditional input formatting |
| Tokenization | T5Tokenizer, max_length=512 | Subword tokenization, padding, truncation |
| Inference A | Greedy (num_beams=1) | Local optimum decoding |
| Inference B | Beam Search (num_beams=5) | Global optimum decoding |
| Evaluation | ROUGE-1, ROUGE-L | Automated NLP evaluation metrics |

### Next Phase: Web Scraper
We will replace the manual `news_article` variable with a function:
```python
def scrape_article(url: str) -> str:
    # Uses requests + BeautifulSoup to fetch article body
    ...
```
**Everything from preprocess_for_t5() onwards stays identical.**  
That's the power of a clean, modular NLP pipeline.